# Replication notebook

Dynamic communities in the Korean political YouTube commenting network.

This notebook reproduces every quantitative result and figure the paper reports.
It contains no analysis logic of its own: each step calls into the `scripts`
package, so the notebook stays readable and the code stays testable. A figure's
plotting code lives directly alongside the computation it draws on (e.g.
`scripts/cohesion.py` has both the cohesion math and its four figure panels) --
there is no separate figures/ tree. Read `scripts/config.py` for the parameters
and `README.md` for setup.

**Before running:** copy `.env.example` to `.env` and fill in the database
credentials, then create and populate the schema:

```
pip install -r requirements.txt
python -m scripts.init_db --create --load data/
```

**Runtime.** The first run streams the whole comment history into a local
Parquet cache and takes tens of minutes; later runs read that cache and take a
few minutes. The polarity queries (step 12) and the LDA fit (step 13) are the
slowest remaining steps.

## 1. Setup

`apply_style()` picks a Korean-capable font -- channel titles and topic keywords
are Korean, and matplotlib has no CJK font of its own. It warns if none is
installed.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, '.')   # run this notebook from the project root

from scripts import (cache, channels, cohesion, communities, config, core_nodes,
                     data, db, network, polarity, style, topics, tracking)

style.apply_style()
connection = db.connect()
print('Connected. Study window:', config.START_DATE, '->', config.END_DATE)

## 2. Parameters

Everything the analysis depends on. Overriding any of these here takes effect
immediately -- the analysis functions read `config` when they are called, not
when they are imported -- so a sensitivity check is a matter of reassigning a
value and re-running the cells below.

In [ ]:
for key in ('START_DATE', 'END_DATE', 'WINDOW_DAYS', 'STEP_DAYS',
            'THETA', 'PHI', 'BETA', 'GRACE',
            'LOUVAIN_RESOLUTION', 'LOUVAIN_SEED',
            'NETWORK_WEIGHT_MODE', 'MIN_EDGE_WEIGHT', 'MIN_COMMUNITY_SIZE',
            'N_TOPICS_GLOBAL', 'TOP_K_DCS'):
    print(f'{key:<22} {getattr(config, key)}')

## 3. Channels and data summary

The channel set is fixed in `scripts/channels.py`. Titles and political leaning
(`wing`) come from the database, not from that file, so a channel with no coded
leaning is dropped here rather than silently entering the analysis with an
unknown side.

In [ ]:
channel_info_map = data.load_channel_info(connection, channels.CHANNEL_IDS)
channel_ids = list(channel_info_map)
print(f'{len(channel_ids)} channels loaded '
      f'(of {len(channels.CHANNEL_IDS)} in the study list)')

In [ ]:
summary_df = data.data_summary(connection, channel_info_map)
print(data.format_summary(summary_df))
style.save_table(summary_df.reset_index(), 'data_summary')
summary_df

## 4. Snapshots

Each snapshot is one non-overlapping 14-day window. Two channels are joined when
someone commented on both inside that window; the edge weight is the symmetric
overlap ratio, which is bounded in [0, 1] and does not simply track channel size
the way a raw shared-commenter count would.

The first call streams the comment history into the Parquet cache under
`cache/`; later runs read straight from it.

In [ ]:
daily_authors = cache.get_daily_authors(channel_ids, config.START_DATE,
                                       config.END_DATE, connection)
print(f'Days with activity: {len(daily_authors)}')

In [ ]:
snapshot_dates, snapshots = network.build_snapshots(daily_authors, channel_ids)

## 5. Community detection

Stage 1 of the two-stage algorithm: Louvain, run independently on each snapshot.
Louvain has no memory between snapshots and its labels are not comparable across
them -- establishing identity over time is stage 2's job.

In [ ]:
snapshot_communities = communities.detect_all(snapshots)

## 6. Tracking: dynamic communities

Stage 2, following Greene, Doyle & Cunningham (2010). A community at snapshot
`t` matches a dynamic community when their Jaccard similarity clears `THETA` or
either containment clears `PHI`. The matching structure between consecutive
snapshots is what produces the event log: birth, death, growth, contraction,
merge, split, and plain continuation.

In [ ]:
dcs, events = tracking.track(snapshot_communities)

In [ ]:
ev_df = tracking.event_dataframe(events, snapshot_dates)
print('Event counts:')
print(ev_df['type'].value_counts().to_string())

log_df = tracking.event_log(events, snapshot_dates, dcs, channel_info_map)
style.save_table(log_df, 'event_log')
log_df.head(20)

## 7. Figure: dynamic-community timeline

One lane per dynamic community over the whole study period. Bar thickness is the
number of member channels; colour is the dominant wing. A confirmed front is
drawn in the wing colour for exactly one window and any remaining stretch is
grey, so time a community survives only on its grace allowance never reads as a
genuine rematch.

In [ ]:
tracking.plot_timeline(dcs, events, snapshot_dates, channel_info_map);

## 8. Figure: static event graph

The same tracking output, compressed for print. Event-free stretches of more
than two snapshots collapse to a break mark and the edge spanning them switches
to a distinct 'cut' style, so a shortened stretch is never mistaken for an
adjacent one. The x-axis is categorical, not a date axis, so quiet months cost
no horizontal space.

In [ ]:
tracking.plot_event_graph(dcs, events, snapshot_dates, channel_info_map);

## 9. Figure: daily comment volume

Context for everything above: how much commenting activity each side carried,
day by day. A spike in structural change around an election means something
different if raw volume spiked with it.

In [ ]:
daily_wing_counts = data.daily_comment_volume(connection, channel_info_map)
data.plot_daily_volume(daily_wing_counts);

## 10. Cohesion: internal density and conductance

Two complementary measurements per (community, snapshot). Internal density asks
how tightly a faction's own members overlap with each other; conductance asks
how separated the faction is from the rest of the network that week. A faction
can be dense and poorly separated, or sparse and well isolated -- the pair is
more informative than either alone.

In [ ]:
cohesion_df, dc_cohesion_summary_df, wing_cohesion_df = cohesion.cohesion_tables(
    dcs, snapshots, snapshot_dates, channel_info_map)

style.save_table(dc_cohesion_summary_df, 'dc_cohesion_summary')
dc_cohesion_summary_df.head(15)

In [ ]:
cohesion.plot_cohesion(cohesion_df, dc_cohesion_summary_df, wing_cohesion_df);

## 11. Critical (core) channels per dynamic community

Two independent rankings inside each faction's own subgraph. Hubs are ranked by
weighted degree: channels whose audience overlaps heavily with the rest of the
faction. Brokers are ranked by betweenness over `1 / weight` distances: channels
that within-faction shortest paths route through. They are deliberately not the
same list -- a broker can have modest degree and still be the channel holding
two sub-groups of the faction together.

In [ ]:
core_df, dc_core_nodes_df = core_nodes.core_nodes_table(
    dcs, snapshots, snapshot_dates, channel_info_map)

style.save_table(dc_core_nodes_df, 'dc_core_nodes')
dc_core_nodes_df.head(20)

In [ ]:
core_nodes.plot_core_nodes(dc_core_nodes_df, dc_cohesion_summary_df, dcs,
                           channel_info_map);

## 12. Figure: polarity distributions

Panel (a) is the baseline: where the whole commenting population sits between
the two political camps. Panels (b) onward repeat the identical measurement
between two factions *of the same wing*, over the window in which both were
active.

The comparison is the finding. If (a) is bimodal and (b) is not, intra-wing
factions share an audience. Bimodality in both means the separation reproduces
itself inside a single political camp.

These queries aggregate per author in SQL and are the slowest step in the
notebook.

In [ ]:
bipartite_scores = polarity.compute_bipartite_polarity(connection, channel_info_map)

In [ ]:
pairs = polarity.resolve_dc_pairs(dcs, dc_cohesion_summary_df, channel_info_map)
print('Same-wing pairs:', pairs)

plottable_pairs, skipped = polarity.polarity_for_pairs(
    connection, dcs, pairs, snapshot_dates, channel_info_map)

In [ ]:
polarity.plot_polarity_distributions(bipartite_scores, plottable_pairs);

## 13. Global LDA topic model

One topic model over every video title and tag blob in the study period. Fitting
a single *global* model rather than one model per community is what makes the
topic axis shared: every faction and every week can then be projected onto the
same topics and compared. Per-community models would each invent their own topic
numbering and could not be placed side by side.

Korean is agglutinative, so tokens come from morphological analysis (Kiwi),
keeping only multi-character common and proper nouns.

In [ ]:
videos_df = cache.get_videos(channel_ids, config.START_DATE, config.END_DATE,
                             connection)
print(f'{len(videos_df):,} videos loaded across '
      f'{videos_df["channel_id"].nunique()} channels')

In [ ]:
model = topics.fit_global_lda(videos_df)

## 14. Figure: DC x topic heatmap

Which topics each faction concentrated on across its lifetime. The x-axis
carries bare topic ids to keep the grid readable; the keyword legend on the
right is what says what each id means.

In [ ]:
topic_matrix, dc_labels = topics.dc_topic_matrix(
    dcs, videos_df, snapshot_dates, model, channel_info_map)

topics.plot_dc_topic_heatmap(topic_matrix, dc_labels, model);

## 15. Figure: topic prevalence over time

The community-agnostic backdrop: what the whole corpus was talking about each
week. A faction's high weight on a topic only means something once you can see
whether every channel was covering it at the time.

In [ ]:
topic_prevalence = topics.topic_prevalence_matrix(snapshot_dates, videos_df, model)
topics.plot_topic_prevalence(topic_prevalence, snapshot_dates, model);

## 16. Cleanup

In [ ]:
connection.close()
print('Figures ->', config.FIGURE_DIR)
print('Tables  ->', config.TABLE_DIR)